## Demo: MoveActivity
The basic steps to set up an OpenCLSim simulation are:
* Import libraries
* Initialise simpy environment
* Define object classes
* Create objects
  * Create sites
  * Create vessels
  * Create activities
* Register processes and run simpy

----

### Under Move activity, we are going to add the object classes including the site objects, vessel objects and activities

#### 0. Import libraries

In [13]:
import datetime, time
import simpy

import pandas as pd

import openclsim.core as core
import openclsim.model as model
import openclsim.plot as plot

#added imports
import shapely.geometry
import pandas as pd

#### 1. Initialise simpy environment

In [3]:
# setup environment
simulation_start = 0
my_env = simpy.Environment(initial_time=simulation_start)

#### 2. Define object classes
This step will be added here

In [10]:
#create a Site object based on desired mixin classes
Site = type (
    "Site",
    (
        core.Identifiable,
        core.Log,
        core.Locatable,
        core.HasContainer,
        core.HasResource,
    ),
    {},
)

#create a TransportProcessingResource object based on desired mixin classes 
TransportProcessingResource = type(
    "TransportProcessingResource",
    (
        core.Identifiable,
        core.ContainerDependentMovable,
        core.HasResource,
        core.Processor,
    ),
    {},
)

#### 3. Create objects
##### 3.1. Create site object(s)
Adding Site objects, vessel objects and activities

In [14]:
# prepare input data for from_site
location_from_site = shapely.geometry.Point(4.18055556, 52.18664444)

# prepare input data for from_site
location_to_site = shapely.geometry.Point(4.25222222, 52.11428333)
data_to_site = {"env": my_env,
                "name": "to_site",
                "geometry": location_to_site,
                "capacity": 100,
                "level": 100
               }
# instantiate to_site 
to_site = Site(**data_to_site)

##### 3.2. Create vessel object(s)

In [15]:
# prepare input data for vessel_01
data_vessel01 = {"env": my_env,
                 "name": "vessel01",
                 "geometry": location_from_site, 
                 "capacity": 5,
                 "compute_v": lambda x: 10
               }
# instantiate vessel_01 
vessel01 = TransportProcessingResource(**data_vessel01)

##### 3.3 Create activity/activities

In [17]:
# initialise registry
registry = {}

In [18]:
# create a basic activity (it just creates an event that shifts time for 'duration')
activity = model.MoveActivity(
    env=my_env,
    name="Move activity",
    registry=registry,
    mover =vessel01,
    destination=to_site,
)

#### 4. Register processes and run simpy

In [19]:
# initate the simpy processes defined in the 'basic activity' and run simpy
model.register_processes([activity])
my_env.run()

#### 5. Inspect results
##### 5.1 Inspect logs
The method plot.get_log_dataframe returns the log of an activity in the form of a dataframe. By adding other activities in a list as the second argument, the Activity can be made more human readable. 

In [24]:
display(plot.get_log_dataframe(activity))

,Activity,Timestamp,ActivityState
0,e063e77c-731e-4c96-8d5c-a5089d8c3b09,1970-01-01 00:00:00.000000,START
1,e063e77c-731e-4c96-8d5c-a5089d8c3b09,1970-01-01 00:15:42.824591,STOP


In [ ]:
display(plot.get_log_dataframe(activity, [activity]))

,Activity,Timestamp,ActivityState
0,Move activity,1970-01-01 00:00:00.000000,START
1,Move activity,1970-01-01 00:15:42.824591,STOP


In [25]:
display(plot.get_log_dataframe(vessel01, [activity]))

,Activity,Timestamp,ActivityState,container level,geometry
0,Move activity,1970-01-01 00:00:00.000000,START,0.0,POINT (4.18055556 52.18664444)
1,Move activity,1970-01-01 00:15:42.824591,STOP,0.0,POINT (4.25222222 52.11428333)


#### 6. Additional logging
The scenario can be extended by adding additional logging instances, that is, the information of the basic activity will also be logged in the activity log of additional activities. This is useful for more complex nested activities. In this example a reporting activity is added where the basic activity log is also added.

In [22]:
# initiate SimPy environment
simulation_start = 0
my_env = simpy.Environment(initial_time=simulation_start)

# create activities
registry = {}
reporting_activity = model.BasicActivity(
    env=my_env,
    name="Reporting activity",
    registry=registry,
    duration=0,
)
basic_activity = model.BasicActivity(
    env=my_env,
    name="Basic activity",
    registry=registry,
    duration=42,
    additional_logs=[reporting_activity],
)

# initate the simpy processes defined in the 'while activity' and run simpy
model.register_processes([basic_activity])
my_env.run()

#### 6.1 Inspect results
We now show the dataframe of the reporting_activity log, and use the basic_activity to make it more human readable.

In [23]:
plot.get_log_dataframe(reporting_activity, [basic_activity])

,Activity,Timestamp,ActivityState,type,ref
0,Basic activity,1970-01-01 00:00:00,START,additional log,7a2255fd-8495-4b28-b45a-d4398070b471
1,Basic activity,1970-01-01 00:00:42,STOP,additional log,7a2255fd-8495-4b28-b45a-d4398070b471
